In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers torch PyMuPDF pandas tqdm fitz tools pymupdf

In [ ]:
import os
import re
import json
import uuid
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

import torch
from transformers import BertTokenizer, BertForSequenceClassification
import pymupdf as fitz  # PyMuPDF

warnings.filterwarnings("ignore")

# ── Paths (adjust BASE_PATH to your Google Drive mount or local path) ──────────
BASE_PATH = Path("/content/drive/Othercomputers/My PC/data_u")
OUTPUT_PATH = Path("/content/drive/MyDrive/IntelliSense/ml_outputs")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
# ── FinBERT model ──────────────────────────────────────────────────────────────
MODEL_NAME = "ProsusAI/finbert"

# ── Chunk settings ─────────────────────────────────────────────────────────────
CHUNK_TOKENS   = 256   # FinBERT max is 512; 256 gives more granular signals
OVERLAP_TOKENS = 32
MIN_SIGNAL_CONFIDENCE = 0.60   # Only surface signals above this threshold

# ── Five Cs keyword mapping ────────────────────────────────────────────────────
# These map FinBERT-negative chunks to specific C categories based on keywords
FIVE_C_KEYWORD_MAP = {
    "Character": [
        "fraud", "misappropriation", "diversion", "siphon", "embezzle",
        "promoter", "governance", "integrity", "whistleblower", "forensic",
        "DRT", "NCLT", "wilful defaulter", "criminal", "fir", "arrest",
        "auditor resign", "qualification", "going concern", "related party",
        "pledge", "insider trading", "SEBI order", "show cause"
    ],
    "Capacity": [
        "revenue decline", "revenue fall", "sales drop", "EBITDA compress",
        "margin pressure", "cash flow stress", "debt service", "DSCR",
        "working capital", "NPA", "non performing", "asset quality",
        "collection efficiency", "overdue", "restructur", "moratorium",
        "liquidity", "ALM", "maturity mismatch"
    ],
    "Capital": [
        "net worth erode", "equity decline", "leverage", "debt equity",
        "capital adequacy", "tier 1", "provisioning", "write off",
        "impairment", "contingent liabilit", "off balance", "subordinated"
    ],
    "Collateral": [
        "collateral", "security", "mortgage", "hypothecat", "pledge",
        "charge", "encumber", "lien", "foreclos", "SARFAESI", "property",
        "valuation", "asset cover", "cross collateral"
    ],
    "Conditions": [
        "regulatory", "RBI circular", "SEBI", "sector headwind", "competition",
        "macro", "interest rate", "inflation", "GDP", "monsoon", "demand slowdown",
        "policy change", "export restrict", "import duty", "supply chain"
    ]
}

print(f"✅ Config loaded | Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"   BASE_PATH  : {BASE_PATH}")
print(f"   OUTPUT_PATH: {OUTPUT_PATH}")


✅ Config loaded | Device: GPU
   BASE_PATH  : /content/drive/Othercomputers/My PC/data_u
   OUTPUT_PATH: /content/drive/MyDrive/IntelliSense/ml_outputs


In [ ]:
# ============================================================
# CELL 3: Load FinBERT
# ============================================================
print("\n⏳ Loading FinBERT from HuggingFace (first run downloads ~440MB)...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model     = BertForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

# FinBERT label order (from ProsusAI repo)
FINBERT_LABELS = {0: "positive", 1: "negative", 2: "neutral"}

print(f"✅ FinBERT loaded on {device}")



⏳ Loading FinBERT from HuggingFace (first run downloads ~440MB)...


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FinBERT loaded on cuda


In [ ]:
# ============================================================
# CELL 4: Core utility functions
# ============================================================

def extract_text_from_pdf(pdf_path: Path, max_pages: int = 80) -> str:
    """
    Extract raw text from PDF using PyMuPDF.
    Limits to max_pages to avoid memory issues on large annual reports.
    """
    try:
        doc = fitz.open(str(pdf_path))
        pages = min(len(doc), max_pages)
        text_chunks = []
        for i in range(pages):
            page = doc[i]
            text_chunks.append(page.get_text("text"))
        doc.close()
        return "\n".join(text_chunks)
    except Exception as e:
        print(f"  ⚠️  PDF read error: {pdf_path.name} → {e}")
        return ""


def chunk_text(text: str, chunk_size: int = CHUNK_TOKENS, overlap: int = OVERLAP_TOKENS):
    """
    Tokenize and split into overlapping windows.
    Returns list of decoded string chunks.
    """
    text = re.sub(r'\s+', ' ', text).strip()
    if not text:
        return []
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks  = []
    start   = 0
    step    = chunk_size - overlap
    while start < len(tokens):
        chunk_tokens = tokens[start : start + chunk_size]
        chunk_text   = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        if len(chunk_text.strip()) > 20:   # skip near-empty chunks
            chunks.append(chunk_text)
        start += step
    return chunks


def classify_chunks(chunks: list[str], batch_size: int = 16) -> list[dict]:
    """
    Run FinBERT on a list of text chunks in batches.
    Returns list of {text, label, confidence, logits}.
    """
    results = []
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        enc   = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        for j, chunk in enumerate(batch):
            label_idx  = int(np.argmax(probs[j]))
            confidence = float(probs[j][label_idx])
            results.append({
                "text"      : chunk,
                "label"     : FINBERT_LABELS[label_idx],
                "confidence": confidence,
                "prob_pos"  : float(probs[j][0]),
                "prob_neg"  : float(probs[j][1]),
                "prob_neu"  : float(probs[j][2]),
            })
    return results


def map_to_five_c(text: str) -> str:
    """Map a text chunk to the most relevant C category based on keyword density."""
    text_lower = text.lower()
    scores = {}
    for c_name, keywords in FIVE_C_KEYWORD_MAP.items():
        scores[c_name] = sum(1 for kw in keywords if kw.lower() in text_lower)
    best_c = max(scores, key=scores.get)
    return best_c if scores[best_c] > 0 else "Character"   # default to Character


def derive_signal_category(text: str) -> str:
    """Map negative news/text to a specific signal_category enum."""
    t = text.lower()
    if any(k in t for k in ["fraud", "scam", "embezzle", "diversion", "fir", "arrest"]):
        return "promoter_fraud_allegation"
    if any(k in t for k in ["drt", "court", "nclt", "arbitration", "legal"]):
        return "company_litigation_news"
    if any(k in t for k in ["default", "npa", "overdue", "restructur"]):
        return "company_financial_stress"
    if any(k in t for k in ["auditor resign", "going concern", "qualification"]):
        return "company_financial_stress"
    if any(k in t for k in ["regulation", "rbi", "sebi", "circular"]):
        return "sector_regulatory_headwind"
    if any(k in t for k in ["promoter", "pledge", "insider"]):
        return "promoter_controversy"
    return "company_financial_stress"

In [ ]:
# ============================================================
# CELL 5: Process PDF documents (Rating Reports + Annual Reports)
# ============================================================

DOC_FOLDERS = {
    "rating_report"  : BASE_PATH / "unstructured" / "rating_reports",
    "annual_report"  : BASE_PATH / "unstructured" / "indian_reports",
    "board_minutes"  : BASE_PATH / "unstructured" / "indian_board",
    "sanction_letter": BASE_PATH / "unstructured" / "sanction_letters",
}

doc_signals_rows = []

for doc_type, folder in DOC_FOLDERS.items():
    if not folder.exists():
        print(f"  ⚠️  Folder not found, skipping: {folder}")
        continue

    pdf_files = list(folder.rglob("*.pdf"))
    print(f"\n📂 {doc_type}: {len(pdf_files)} PDFs found")

    for pdf_path in tqdm(pdf_files, desc=doc_type):
        # ── Derive company_id from filename or parent folder name ──────────
        # Convention: parent folder name = company ticker, or COMP_XXXXX prefix
        company_id  = pdf_path.parent.name if pdf_path.parent != folder else pdf_path.stem
        source_doc  = pdf_path.name

        # Extract and chunk text
        raw_text = extract_text_from_pdf(pdf_path, max_pages=60)
        if not raw_text.strip():
            continue
        chunks = chunk_text(raw_text)
        if not chunks:
            continue

        # Classify
        classifications = classify_chunks(chunks)

        # Keep only high-confidence negative signals
        for idx, result in enumerate(classifications):
            if result["label"] == "negative" and result["confidence"] >= MIN_SIGNAL_CONFIDENCE:
                c_cat    = map_to_five_c(result["text"])
                sig_cat  = derive_signal_category(result["text"])
                doc_signals_rows.append({
                    "signal_id"              : str(uuid.uuid4()),
                    "company_id"             : company_id,
                    "source_document"        : source_doc,
                    "doc_type"               : doc_type,
                    "chunk_index"            : idx,
                    "relevant_text_chunk"    : result["text"][:500],
                    "finbert_label"          : result["label"],
                    "finbert_confidence"     : round(result["confidence"], 4),
                    "prob_positive"          : round(result["prob_pos"], 4),
                    "prob_negative"          : round(result["prob_neg"], 4),
                    "prob_neutral"           : round(result["prob_neu"], 4),
                    "feeds_into_c"           : c_cat,
                    "signal_category"        : sig_cat,
                    "severity_score"         : round(result["confidence"], 4),
                    "is_high_severity"       : result["confidence"] > 0.80,
                    "processed_timestamp"    : datetime.utcnow().isoformat(),
                })

print(f"\n✅ Document processing complete | Total negative signals: {len(doc_signals_rows)}")



📂 rating_report: 10 PDFs found


rating_report: 100%|██████████| 10/10 [00:40<00:00,  4.05s/it]



📂 annual_report: 99 PDFs found


annual_report: 100%|██████████| 99/99 [12:30<00:00,  7.58s/it]



📂 board_minutes: 1014 PDFs found


...

  ⚠️  PDF read error: ADVANCE_14112025200008_outcome_of_BM_14112025.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/ADVANCE_14112025200008_outcome_of_BM_14112025.pdf'.
  ⚠️  PDF read error: ALPHAGEO_11082023152030_Outcomeofboardmeeting11082023.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/ALPHAGEO_11082023152030_Outcomeofboardmeeting11082023.pdf'.
  ⚠️  PDF read error: AMANTA_26092025170049_outcome.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/AMANTA_26092025170049_outcome.pdf'.
  ⚠️  PDF read error: AMBICAAGAR_28052025205459_Ambicaresultsfinal.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/AMBICAAGAR_28052025205459_Ambicaresultsfinal.pdf'.


board_minutes:   1%|          | 7/1014 [00:07<14:38,  1.15it/s]

  ⚠️  PDF read error: ASHOKLEY_14082025124846_ALResultsjune2025.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/ASHOKLEY_14082025124846_ALResultsjune2025.pdf'.


board_minutes:  11%|█         | 108/1014 [10:42<1:29:48,  5.95s/it]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL 5b — Run ONLY remaining folders (rating + annual already done)
# ============================================================

# Only process what hasn't run yet
REMAINING_FOLDERS = {
    "board_minutes"  : BASE_PATH / "unstructured" / "indian_board",
    "sanction_letter": BASE_PATH / "unstructured" / "sanction_letters",
}

DEMO_LIMITS = {
    "board_minutes"  : 200,
    "sanction_letter": None,
}

for doc_type, folder in REMAINING_FOLDERS.items():
    if not folder.exists():
        print(f"  ⚠️  Folder not found, skipping: {folder}")
        continue

    pdf_files = list(folder.rglob("*.pdf"))
    limit = DEMO_LIMITS.get(doc_type)
    if limit:
        pdf_files = pdf_files[:limit]

    print(f"\n📂 {doc_type}: processing {len(pdf_files)} PDFs")

    for pdf_path in tqdm(pdf_files, desc=doc_type):
        company_id = pdf_path.parent.name if pdf_path.parent != folder else pdf_path.stem
        source_doc = pdf_path.name
        raw_text   = extract_text_from_pdf(pdf_path, max_pages=60)
        if not raw_text.strip():
            continue
        chunks = chunk_text(raw_text)
        if not chunks:
            continue
        classifications = classify_chunks(chunks)
        for idx, result in enumerate(classifications):
            if result["label"] == "negative" and result["confidence"] >= MIN_SIGNAL_CONFIDENCE:
                c_cat   = map_to_five_c(result["text"])
                sig_cat = derive_signal_category(result["text"])
                doc_signals_rows.append({
                    "signal_id"           : str(uuid.uuid4()),
                    "company_id"          : company_id,
                    "source_document"     : source_doc,
                    "doc_type"            : doc_type,
                    "chunk_index"         : idx,
                    "relevant_text_chunk" : result["text"][:500],
                    "finbert_label"       : result["label"],
                    "finbert_confidence"  : round(result["confidence"], 4),
                    "prob_positive"       : round(result["prob_pos"], 4),
                    "prob_negative"       : round(result["prob_neg"], 4),
                    "prob_neutral"        : round(result["prob_neu"], 4),
                    "feeds_into_c"        : c_cat,
                    "signal_category"     : sig_cat,
                    "severity_score"      : round(result["confidence"], 4),
                    "is_high_severity"    : result["confidence"] > 0.80,
                    "processed_timestamp" : datetime.utcnow().isoformat(),
                })

print(f"\n✅ Remaining folders done | Total signals so far: {len(doc_signals_rows)}")


📂 board_minutes: processing 200 PDFs


board_minutes:   0%|          | 0/200 [00:00<?, ?it/s]

  ⚠️  PDF read error: ADVANCE_14112025200008_outcome_of_BM_14112025.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/ADVANCE_14112025200008_outcome_of_BM_14112025.pdf'.
  ⚠️  PDF read error: ALPHAGEO_11082023152030_Outcomeofboardmeeting11082023.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/ALPHAGEO_11082023152030_Outcomeofboardmeeting11082023.pdf'.
  ⚠️  PDF read error: AMANTA_26092025170049_outcome.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/AMANTA_26092025170049_outcome.pdf'.
  ⚠️  PDF read error: AMBICAAGAR_28052025205459_Ambicaresultsfinal.pdf → Cannot open empty file: filename='/content/drive/Othercomputers/My PC/data_u/unstructured/indian_board/AMBICAAGAR_28052025205459_Ambicaresultsfinal.pdf'.
  ⚠️  PDF read error: ASHOKLEY_14082025124846_ALResultsjune2025.pdf → Cannot open empty file: filename

board_minutes: 100%|██████████| 200/200 [10:10<00:00,  3.05s/it]



📂 sanction_letter: processing 30 PDFs


sanction_letter: 100%|██████████| 30/30 [00:15<00:00,  1.92it/s]


✅ Remaining folders done | Total signals so far: 440


In [ ]:
# ============================================================
# CELL 6: Process News Articles CSV
# ============================================================

news_csv = BASE_PATH / "external intelligence" / "news_intelligence" / "news_articles_crawled.csv"
news_signal_rows = []

if news_csv.exists():
    news_df = pd.read_csv(news_csv)
    print(f"\n📰 News articles loaded: {len(news_df)} rows")

    # Combine headline + full text for classification
    # Truncate full_text to avoid extremely long articles
    def prepare_news_text(row):
        headline = str(row.get("article_headline", ""))
        body     = str(row.get("article_full_text", ""))[:2000]   # first 2000 chars
        return f"{headline}. {body}".strip()

    # Process in batches
    news_texts = [prepare_news_text(r) for _, r in news_df.iterrows()]
    # Chunk each article independently (most fit in 1-2 chunks)
    all_article_chunks = []   # list of (original_row_idx, chunk_text)
    for i, text in enumerate(news_texts):
        for chunk in chunk_text(text):
            all_article_chunks.append((i, chunk))

    print(f"   Total news chunks to classify: {len(all_article_chunks)}")

    # Classify in one pass
    chunk_texts_only = [c[1] for c in all_article_chunks]
    classifications  = classify_chunks(chunk_texts_only, batch_size=32)

    # Re-join with original row metadata
    for (orig_idx, chunk_str), result in zip(all_article_chunks, classifications):
        if result["label"] == "negative" and result["confidence"] >= MIN_SIGNAL_CONFIDENCE:
            orig_row   = news_df.iloc[orig_idx]
            c_cat      = map_to_five_c(chunk_str)
            sig_cat    = derive_signal_category(chunk_str)
            news_signal_rows.append({
                "signal_id"           : str(uuid.uuid4()),
                "article_id"          : str(orig_row.get("article_id", f"ART_{orig_idx:06d}")),
                "company_id"          : str(orig_row.get("company_id", "")),
                "company_cin"         : str(orig_row.get("company_cin", "")),
                "article_url"         : str(orig_row.get("article_url", "")),
                "source_publication"  : str(orig_row.get("source_publication", "")),
                "published_date"      : str(orig_row.get("published_date", "")),
                "signal_category"     : sig_cat,
                "severity_score"      : round(result["confidence"], 4),
                "is_high_severity"    : result["confidence"] > 0.80,
                "relevant_text_chunk" : chunk_str[:500],
                "finbert_risk_category": result["label"],
                "feeds_into_c"        : c_cat,
                "processed_timestamp" : datetime.utcnow().isoformat(),
            })

    print(f"✅ News signals extracted: {len(news_signal_rows)}")
else:
    print(f"  ⚠️  news_articles_crawled.csv not found at {news_csv}")



📰 News articles loaded: 16662 rows
   Total news chunks to classify: 35963
✅ News signals extracted: 12129


In [ ]:
# ============================================================
# CELL 7: Save outputs
# ============================================================

# 1) Document signals (from PDFs)
doc_signals_df = pd.DataFrame(doc_signals_rows)
doc_out = OUTPUT_PATH / "finbert_doc_signals.csv"
doc_signals_df.to_csv(doc_out, index=False)
print(f"\n💾 Saved: {doc_out}  ({len(doc_signals_df)} rows)")

# 2) News risk signals (matches rulebook table: news_risk_signals)
news_signals_df = pd.DataFrame(news_signal_rows)
news_out = OUTPUT_PATH / "news_risk_signals.csv"
news_signals_df.to_csv(news_out, index=False)
print(f"💾 Saved: {news_out}  ({len(news_signals_df)} rows)")

# 3) Per-company summary roll-up
all_signals = pd.concat([
    doc_signals_df[["company_id", "feeds_into_c", "severity_score", "is_high_severity", "signal_category"]],
    news_signals_df[["company_id", "feeds_into_c", "severity_score", "is_high_severity", "signal_category"]],
], ignore_index=True)

if not all_signals.empty:
    summary = all_signals.groupby("company_id").agg(
        total_negative_signals   = ("severity_score", "count"),
        avg_severity             = ("severity_score", "mean"),
        max_severity             = ("severity_score", "max"),
        high_severity_count      = ("is_high_severity", "sum"),
        primary_c_category       = ("feeds_into_c", lambda x: x.mode()[0] if len(x) > 0 else "Unknown"),
        dominant_signal_type     = ("signal_category", lambda x: x.mode()[0] if len(x) > 0 else "Unknown"),
    ).reset_index()
    summary["finbert_risk_score"] = (
        summary["avg_severity"] * 0.4 +
        (summary["high_severity_count"] / summary["total_negative_signals"].clip(1)) * 0.6
    ).round(4)
    summary_out = OUTPUT_PATH / "finbert_summary.csv"
    summary.to_csv(summary_out, index=False)
    print(f"💾 Saved: {summary_out}  ({len(summary)} companies summarized)")




💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/finbert_doc_signals.csv  (440 rows)
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/news_risk_signals.csv  (12129 rows)
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/finbert_summary.csv  (703 companies summarized)


In [ ]:
# ============================================================
# CELL 8: Ugro Capital spot-check (demo validation)
# ============================================================
print("\n" + "="*60)
print("UGRO CAPITAL SPOT-CHECK")
print("="*60)

ugro_signals = all_signals[
    all_signals["company_id"].str.upper().str.contains("UGRO", na=False)
]

if ugro_signals.empty:
    print("  No Ugro-tagged signals found. Check company_id values in source CSVs.")
    print("  Tip: Run: news_df['company_id'].value_counts().head(20) to see IDs")
else:
    print(f"  Total signals : {len(ugro_signals)}")
    print(f"  High severity : {ugro_signals['is_high_severity'].sum()}")
    print(f"\n  By C Category:")
    print(ugro_signals["feeds_into_c"].value_counts().to_string())
    print(f"\n  By Signal Type:")
    print(ugro_signals["signal_category"].value_counts().to_string())
    print(f"\n  Top 3 high-severity excerpts:")
    top3 = ugro_signals.nlargest(3, "severity_score")
    for _, row in top3.iterrows():
        print(f"\n  [{row['feeds_into_c']} | {row['severity_score']:.3f}]")
        print(f"  {row.get('relevant_text_chunk', '')[:200]}...")



UGRO CAPITAL SPOT-CHECK
  No Ugro-tagged signals found. Check company_id values in source CSVs.
  Tip: Run: news_df['company_id'].value_counts().head(20) to see IDs


In [ ]:
# ============================================================
# CELL 9: Output schema preview (for Five Cs integration)
# ============================================================
print("\n" + "="*60)
print("OUTPUT SCHEMA — feeds directly into Five Cs Layer")
print("="*60)
print("""
finbert_doc_signals.csv
  signal_id, company_id, source_document, doc_type,
  chunk_index, relevant_text_chunk, finbert_label,
  finbert_confidence, prob_positive, prob_negative, prob_neutral,
  feeds_into_c, signal_category, severity_score, is_high_severity

news_risk_signals.csv   ← MATCHES RULEBOOK TABLE EXACTLY
  signal_id, article_id, company_id, company_cin,
  article_url, source_publication, published_date,
  signal_category, severity_score, is_high_severity,
  relevant_text_chunk, finbert_risk_category, feeds_into_c

finbert_summary.csv   ← Roll-up for Five Cs Scorecard
  company_id, total_negative_signals, avg_severity, max_severity,
  high_severity_count, primary_c_category, dominant_signal_type,
  finbert_risk_score (0.0-1.0, used in Character C weighting)

HOW IT FEEDS FIVE Cs:
  Character C   ← signals where feeds_into_c = 'Character'
                   + finbert_risk_score from summary
  Capacity C    ← signals where feeds_into_c = 'Capacity'
  Conditions C  ← news signals where signal_category contains 'sector_'
""")

print("✅ FinBERT pipeline complete.")



OUTPUT SCHEMA — feeds directly into Five Cs Layer

finbert_doc_signals.csv
  signal_id, company_id, source_document, doc_type,
  chunk_index, relevant_text_chunk, finbert_label,
  finbert_confidence, prob_positive, prob_negative, prob_neutral,
  feeds_into_c, signal_category, severity_score, is_high_severity

news_risk_signals.csv   ← MATCHES RULEBOOK TABLE EXACTLY
  signal_id, article_id, company_id, company_cin,
  article_url, source_publication, published_date,
  signal_category, severity_score, is_high_severity,
  relevant_text_chunk, finbert_risk_category, feeds_into_c

finbert_summary.csv   ← Roll-up for Five Cs Scorecard
  company_id, total_negative_signals, avg_severity, max_severity,
  high_severity_count, primary_c_category, dominant_signal_type,
  finbert_risk_score (0.0-1.0, used in Character C weighting)

HOW IT FEEDS FIVE Cs:
  Character C   ← signals where feeds_into_c = 'Character'
                   + finbert_risk_score from summary
  Capacity C    ← signals where fee

In [ ]:
print(f"Companies with signals: {news_signals_df['company_id'].nunique()}")
print(f"Avg signals per company: {len(news_signals_df)/news_signals_df['company_id'].nunique():.1f}")
print(f"\nTop signal categories:")
print(news_signals_df["signal_category"].value_counts())
print(f"\nFeeds into C breakdown:")
print(news_signals_df["feeds_into_c"].value_counts())


Companies with signals: 647
Avg signals per company: 18.7

Top signal categories:
signal_category
company_financial_stress      6949
promoter_fraud_allegation     4073
company_litigation_news        797
sector_regulatory_headwind     218
promoter_controversy            92
Name: count, dtype: int64

Feeds into C breakdown:
feeds_into_c
Character     9596
Collateral     882
Conditions     819
Capacity       778
Capital         54
Name: count, dtype: int64


In [ ]:
# ============================================================
# Save FinBERT model + tokenizer locally to Drive
# ============================================================
FINBERT_SAVE_PATH = OUTPUT_PATH /"finbert_model"/ "finbert_model"
FINBERT_SAVE_PATH.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(FINBERT_SAVE_PATH))
tokenizer.save_pretrained(str(FINBERT_SAVE_PATH))

print(f"✅ FinBERT saved to: {FINBERT_SAVE_PATH}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FinBERT saved to: /content/drive/MyDrive/IntelliSense/ml_outputs/finbert_model/finbert_model
